# Deutsch–Jozsa Algorithm — Qiskit Hackathon Solution

This notebook addresses **Problem Statement 3** from the Qiskit Fall Fest 2026 handout.

It implements the Deutsch–Jozsa algorithm for an `n`-qubit input register, provides **two constant oracles** and **two balanced oracles**, draws the circuits, runs simulations, checks the measured result, and compares oracle-query complexity.

> **Important:** the algorithm's exact guarantee assumes the oracle is promised to be either constant or balanced. The simulation uses an ideal quantum simulator; real hardware noise can introduce incorrect measurement outcomes.

## 1. Install the required packages

Run this cell once in your Jupyter environment. If Qiskit is already installed, it will simply ensure the packages are available.

In [ ]:
%pip install -q qiskit qiskit-aer matplotlib

## 2. Imports and configuration

We use Qiskit's `QuantumCircuit` to build the circuits and `AerSimulator` to simulate measurements. The default is `n = 3` input qubits; the balanced oracle examples require at least two.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
import matplotlib.pyplot as plt
from math import ceil, log2

N_INPUTS = 3
SHOTS = 1024

if N_INPUTS < 2:
    raise ValueError("Choose N_INPUTS >= 2 to demonstrate both balanced oracle examples.")

simulator = AerSimulator()
print(f"Input qubits: {N_INPUTS}")
print(f"Measurement shots per circuit: {SHOTS}")

## 3. Build the Deutsch–Jozsa circuit

The circuit uses `n` input qubits and one ancilla/output qubit.

1. Initialize input qubits to `|0...0>` and the ancilla to `|1>`.
2. Apply Hadamard gates to every qubit.
3. Apply the selected oracle.
4. Apply Hadamard gates to the input qubits again.
5. Measure only the input register.

Oracle choices in this notebook:

- **Constant 0:** `f(x) = 0`
- **Constant 1:** `f(x) = 1`
- **Balanced A:** `f(x) = x[0]`
- **Balanced B:** `f(x) = x[0] XOR x[1]`

The last two functions are balanced because their outputs are 0 for exactly half of all inputs and 1 for the other half.

In [ ]:
def apply_oracle(qc, n, oracle_name):
    """Apply one of the four Deutsch–Jozsa oracles to qc.

    Input qubits are q[0] ... q[n-1].
    The ancilla is q[n].
    """
    ancilla = n

    if oracle_name == "constant_0":
        # f(x) = 0: do nothing.
        pass

    elif oracle_name == "constant_1":
        # f(x) = 1: flip the ancilla for every input.
        # With the ancilla in |->, this changes only a global phase.
        qc.x(ancilla)

    elif oracle_name == "balanced_x0":
        # f(x) = x[0]: toggle the ancilla when input bit 0 is 1.
        qc.cx(0, ancilla)

    elif oracle_name == "balanced_x0_xor_x1":
        # f(x) = x[0] XOR x[1].
        # Toggle the ancilla for each input bit equal to 1.
        qc.cx(0, ancilla)
        qc.cx(1, ancilla)

    else:
        raise ValueError(f"Unknown oracle: {oracle_name}")


def deutsch_jozsa_circuit(n, oracle_name, draw_measurements=True):
    """Create a Deutsch–Jozsa circuit for n input qubits."""
    qc = QuantumCircuit(n + 1, n)

    # Step 1: input register |0...0>, ancilla |1>.
    qc.x(n)

    # Step 2: Hadamard on all input qubits and ancilla.
    for q in range(n + 1):
        qc.h(q)

    # Step 3: one oracle query.
    apply_oracle(qc, n, oracle_name)

    # Step 4: Hadamard on the input register only.
    for q in range(n):
        qc.h(q)

    # Step 5: measure the input register.
    if draw_measurements:
        for q in range(n):
            qc.measure(q, q)

    return qc


ORACLES = {
    "constant_0": "Constant oracle: f(x) = 0",
    "constant_1": "Constant oracle: f(x) = 1",
    "balanced_x0": "Balanced oracle: f(x) = x[0]",
    "balanced_x0_xor_x1": "Balanced oracle: f(x) = x[0] XOR x[1]",
}

for name, description in ORACLES.items():
    print(f"{name}: {description}")

## 4. Display the circuit diagrams

The circuit diagrams show the Hadamard gates, oracle operation, and final measurements. In each circuit, the last qubit is the ancilla.

In [ ]:
for oracle_name, description in ORACLES.items():
    print("\\n" + "=" * 80)
    print(description)
    circuit = deutsch_jozsa_circuit(N_INPUTS, oracle_name)
    display(circuit.draw("mpl", fold=-1))

## 5. Run the simulations and verify the results

For a constant oracle, the measured input string must be all zeros. For a balanced oracle, the measured string must be non-zero.

The simulator may display bitstrings in reverse order relative to qubit indices because Qiskit prints classical bit 0 on the right. This does not affect the all-zero versus non-zero test.

In [ ]:
def expected_classification(oracle_name):
    return "constant" if oracle_name.startswith("constant") else "balanced"


def run_deutsch_jozsa(n, oracle_name, shots=1024, show_circuit=False):
    circuit = deutsch_jozsa_circuit(n, oracle_name)
    compiled = transpile(circuit, simulator)
    result = simulator.run(compiled, shots=shots).result()
    counts = result.get_counts()

    # The algorithm's decision depends only on whether the result is all zeros.
    observed = "constant" if set(counts.keys()) == {"0" * n} else "balanced"

    # Any nonzero bitstring indicates balanced. For a constant oracle, only 0...0 should occur.
    # With multiple shots, check whether every observed key is consistent with the expected answer.
    if expected_classification(oracle_name) == "constant":
        correct = all(bitstring == "0" * n for bitstring in counts)
    else:
        correct = all(bitstring != "0" * n for bitstring in counts)

    return {
        "oracle": oracle_name,
        "expected": expected_classification(oracle_name),
        "observed": observed,
        "counts": counts,
        "correct": correct,
        "circuit": circuit,
    }


results = []
for oracle_name in ORACLES:
    result = run_deutsch_jozsa(N_INPUTS, oracle_name, SHOTS)
    results.append(result)
    print(f"\n{ORACLES[oracle_name]}")
    print("Measurement counts:", result["counts"])
    print("Expected classification:", result["expected"])
    print("Observed classification:", result["observed"])
    print("Verification passed:", result["correct"])

assert all(result["correct"] for result in results), "At least one oracle did not pass verification."
print("\nAll four oracle tests passed.")

## 6. Visualize the measurement counts

In an ideal simulator, constant oracles produce only the all-zero result. Balanced oracles produce non-zero results. The exact non-zero bitstring depends on the balanced function.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.flatten()

for ax, result in zip(axes, results):
    counts = result["counts"]
    keys = list(counts.keys())
    values = [counts[k] for k in keys]
    ax.bar(keys, values)
    ax.set_title(result["oracle"])
    ax.set_xlabel("Measured input bitstring")
    ax.set_ylabel("Counts")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 7. Compare quantum and classical oracle-query complexity

For `n` input bits:

- **Deutsch–Jozsa quantum algorithm:** one oracle query.
- **Deterministic classical algorithm, worst case:** `2^(n-1) + 1` queries.

Why does the classical worst case need this many? If the first `2^(n-1)` distinct inputs all return the same value, the function could still be balanced. One additional query forces the conclusion under the promise.

This comparison counts **oracle queries**, not the full runtime or hardware cost.

In [ ]:
n = N_INPUTS
quantum_queries = 1
classical_worst_case = 2 ** (n - 1) + 1

print(f"Number of input bits (n): {n}")
print(f"Quantum oracle queries: {quantum_queries}")
print(f"Classical deterministic worst-case queries: {classical_worst_case}")
print(f"Classical-to-quantum query ratio: {classical_worst_case / quantum_queries:.0f}:1")

ns = list(range(1, 11))
classical_queries = [2 ** (k - 1) + 1 for k in ns]
quantum_queries_line = [1 for _ in ns]

plt.figure(figsize=(8, 5))
plt.plot(ns, classical_queries, marker="o", label="Classical deterministic worst case")
plt.plot(ns, quantum_queries_line, marker="s", label="Deutsch–Jozsa quantum queries")
plt.yscale("log", base=2)
plt.xlabel("Number of input bits (n)")
plt.ylabel("Oracle queries (log scale)")
plt.title("Oracle-query complexity")
plt.grid(True, which="both", linestyle=":")
plt.legend()
plt.show()

## 8. Interpretation and limitations

- **Superposition:** Hadamard gates place the input register into a superposition of all possible input strings.
- **Phase kickback:** The ancilla starts in `|->`; the oracle encodes `f(x)` into the phase of each input component.
- **Interference:** The second set of Hadamard gates makes amplitudes combine so that the all-zero result occurs with certainty for constant functions and has zero probability for balanced functions.
- **Promise:** The decision rule is valid only when the oracle is guaranteed to be constant or balanced.
- **Noise:** These results come from an ideal simulator. On real quantum hardware, gate errors and measurement noise can cause occasional incorrect results.
- **Oracle query vs. circuit cost:** One oracle query does not mean the entire circuit requires only one gate. The implementation of the oracle itself may have a cost.

## 9. Conclusion

The notebook implements four test oracles (two constant and two balanced), displays their circuits, simulates measurements, verifies the classifications, visualizes the results, and compares query complexity. These directly address the main deliverables in Problem Statement 3.